# Extra credit — Logistic Regression

TrafficSliver로 5개 경로에 분할된 트래픽(split)에서 LR을 평가한다. 조건·평가 방식은 모델팀 공통(SVM·RF와 같음)이다.

| 조건 | 학습 | 평가 |
| --- | --- | --- |
| A | 메인 train | extra split test |
| B | extra split train | extra split test |
| C | 메인 train + extra split train | extra split test |

- 빈 행 포함(`include`) / 제외(`exclude`) 버전 각각 실행 (A는 한 번 학습 후 두 버전 test에 평가)
- 분할: `splits/extra_split_seed42.csv` (메인 `split_seed42.csv`와 맞춘 방문 단위 분할, `extra_credit/make_extra_split.ipynb`)
- 필요한 파일: `data/extra/features_extra_split_trim0.csv.gz` (루트 README의 데이터 준비 참고)
- 하이퍼파라미터: 메인 best 값 고정 (Closed C=100, 이진 C=1000)
- 주 지표: 방문 단위 정확도 (한 방문의 split 확률 평균 → argmax)

CPU로 실행하며 전체 약 1시간 걸린다(Colab CPU 기준). 결과는 조건·버전마다 끝나는 대로 저장하고, 다시 실행하면 이미 저장된 조건은 건너뛴다.

## 0. 설정

In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_repo_root():
    """저장소 루트(splits/ 폴더가 있는 위치)를 찾는다. Colab에서 노트북만 연 경우 저장소를 내려받는다."""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "splits" / "split_seed42.csv").exists():
            return p
    if "google.colab" in sys.modules:
        dst = Path("/content/website-fingerprinting")
        if not dst.exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/EWHA-MLB/website-fingerprinting.git", str(dst)], check=True)
        return dst
    raise FileNotFoundError("저장소 안에서 실행해야 한다 (splits/ 폴더를 찾지 못함)")

ROOT       = find_repo_root()
EC_FILE    = str(ROOT / "data" / "extra" / "features_extra_split_trim0.csv.gz")   # 빈 split 포함 버전
MAIN_MON   = str(ROOT / "features" / "기본 실험" / "features_mon_trim0.csv")
MAIN_UNMON = str(ROOT / "features" / "기본 실험" / "features_unmon10k_trim0.csv")
SPLIT_MAIN = str(ROOT / "splits" / "split_seed42.csv")
SPLIT_EC   = str(ROOT / "splits" / "extra_split_seed42.csv")
OUT_DIR    = str(ROOT / "results" / "logistic_regression" / "extra_credit")

LR_C       = {"closed": 100, "binary": 1000}   # 메인 best 값
MAX_ITER   = 5000
SEED       = 42
RUN_BINARY = False      # Open-world 이진은 선택. 필요하면 True
SKIP_IF_EXISTS = True   # 이미 저장된 조건·버전은 건너뜀


In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
for p in [EC_FILE, MAIN_MON, MAIN_UNMON, SPLIT_MAIN, SPLIT_EC]:
    print("있음" if os.path.exists(p) else "없음 → 경로 확인", "|", p)

In [ ]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 수렴 여부는 note의 n_iter로 확인

## 1. 데이터 읽기

특징 110개만 입력으로 쓰고, 분할은 extra는 `source_group_id`, 메인은 `record_id`로 붙인다.

빈 split 제외 버전은 포함 버전 파일에서 `status == "empty"` 행을 빼서 만든다.

In [ ]:
G = ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")

ec_split = pd.read_csv(SPLIT_EC)[["source_group_id", "split"]]
d = pd.read_csv(EC_FILE).merge(ec_split, on="source_group_id", how="left")
assert d.split.notna().all(), "분할 파일에 없는 방문이 있음"
d = d.sort_values("record_id").reset_index(drop=True)
# 제외 버전 = 포함 버전에서 빈 split(패킷 0개, 특징이 모두 0) 행을 뺀 것. 데이터팀 *_without_zero 파일과 같은 172,085행
ec = {"include": d, "exclude": d[d.status != "empty"].reset_index(drop=True)}
assert len(ec["exclude"]) == 172_085

FEATS = [c for c in ec["include"].columns if c.startswith(G)]
assert len(FEATS) == 110, len(FEATS)

main_split = pd.read_csv(SPLIT_MAIN).set_index("record_id").split
main = pd.concat([pd.read_csv(MAIN_MON), pd.read_csv(MAIN_UNMON)], ignore_index=True)
main["split"] = main.record_id.map(main_split)
assert main.split.notna().all() and all(c in main.columns for c in FEATS)
main_tr = main[main.split == "train"].sort_values("record_id").reset_index(drop=True)

for ver, d in ec.items():
    print(f"[{ver}] {len(d):,}행 | train {(d.split == 'train').sum():,} / test {(d.split == 'test').sum():,} | "
          f"mon {(d.dataset == 'mon').sum():,} / unmon {(d.dataset == 'unmon').sum():,}")
print(f"[메인 train] mon {(main_tr.dataset == 'mon').sum():,} / unmon {(main_tr.dataset != 'mon').sum():,}")   # 메인 unmon의 dataset 값은 'unmon10k'

[include] 190,000행 | train 152,615 / test 37,385 | mon 95,000 / unmon 95,000
[exclude] 172,085행 | train 138,050 / test 34,035 | mon 87,038 / unmon 85,047
[메인 train] mon 15,200 / unmon 8,000


## 2. 실험 함수

- `task_view`: 과제별 데이터. Closed는 mon만(정답 = `label`), 이진은 mon 1 / unmon -1
- `run`: 조건·버전 하나를 학습·평가하고 `ec_test_proba_{task}_{condition}_{version}.csv`와 `ec_summary.csv`에 저장
- 행 단위 `row_acc_empty_wrong`: 제외 버전에서 빠진 빈 행을 오답으로 센 값 (분모 = 포함 버전 test 행 수)

In [ ]:
SUMMARY = os.path.join(OUT_DIR, "ec_summary.csv")
KEYS = ["model", "task", "condition", "version", "features"]

def task_view(df, task):
    if task == "closed":
        d = df[df.dataset == "mon"].copy(); d["y"] = d.label
    else:
        d = df.copy(); d["y"] = np.where(d.dataset == "mon", 1, -1)
    return d

def make_lr(C):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=MAX_ITER, random_state=SEED))

def train_data(task, cond, ver):
    parts = []
    if cond in ("A", "C"):
        parts.append(task_view(main_tr, task))
    if cond in ("B", "C"):
        d = task_view(ec[ver], task); parts.append(d[d.split == "train"])
    return pd.concat(parts, ignore_index=True)

def load_summary():
    if not os.path.exists(SUMMARY):
        return pd.DataFrame(columns=KEYS)
    s = pd.read_csv(SUMMARY)
    for c in ["n_train_rows", "n_test_rows", "n_test_visits"]:
        s[c] = s[c].astype(int)
    return s

def save_row(row):
    s = load_summary()
    mask = np.ones(len(s), bool)
    for k in KEYS:
        mask &= s[k].astype(str).to_numpy() == str(row[k])
    s = pd.concat([s[~mask], pd.DataFrame([row])], ignore_index=True)
    s.to_csv(SUMMARY, index=False)

def proba_path(task, cond, ver):
    return os.path.join(OUT_DIR, f"ec_test_proba_{task}_{cond}_{ver}.csv")

_model_A = {}   # 조건 A 모델은 버전과 무관하므로 한 번만 학습

def run(task, cond, ver):
    path = proba_path(task, cond, ver)
    s = load_summary()
    done = ((s.task == task) & (s.condition == cond) & (s.version == ver)).any() if len(s) else False
    if SKIP_IF_EXISTS and done and os.path.exists(path):
        print(f"[{task} {cond} {ver}] 이미 있음 → 건너뜀"); return

    te = task_view(ec[ver], task); te = te[te.split == "test"].reset_index(drop=True)
    tr = train_data(task, cond, ver)

    t0 = time.time()
    if cond == "A" and task in _model_A:
        model, fit_sec, n_iter = _model_A[task]
    else:
        model = make_lr(LR_C[task]).fit(tr[FEATS], tr.y)
        fit_sec, n_iter = time.time() - t0, int(np.max(model[-1].n_iter_))
        if cond == "A":
            _model_A[task] = (model, fit_sec, n_iter)

    classes = model.classes_
    P = model.predict_proba(te[FEATS])
    cols = [f"p_{c}" for c in classes]
    out = pd.concat([te[["record_id", "source_group_id", "variant", "y"]], pd.DataFrame(P, columns=cols)], axis=1)
    out.to_csv(path, index=False, float_format="%.6g")

    # 행 단위
    pred = classes[P.argmax(1)]
    n_correct = int((pred == te.y.to_numpy()).sum())
    n_inc = len(task_view(ec["include"], task).query("split == 'test'"))
    # 방문 단위: 방문의 split 확률 평균
    v = out.groupby("source_group_id")[cols].mean()
    v_y = out.groupby("source_group_id").y.first().loc[v.index].to_numpy()
    v_pred = classes[v.to_numpy().argmax(1)]

    row = {"model": "lr", "task": task, "condition": cond, "version": ver, "features": "all110",
           "n_train_rows": len(tr), "n_test_rows": len(te), "n_test_visits": len(v),
           "visit_acc": (v_pred == v_y).mean(),
           "visit_macro_f1": f1_score(v_y, v_pred, average="macro"),
           "row_acc": n_correct / len(te),
           "row_acc_empty_wrong": n_correct / n_inc,
           "fit_sec": round(fit_sec, 1),
           "note": f"C={LR_C[task]}, n_iter={n_iter}" + (" (미수렴)" if n_iter >= MAX_ITER else "") + (", A 모델 재사용" if cond == "A" and ver == "exclude" else "")}
    save_row(row)
    print(f"[{task} {cond} {ver}] 학습 {len(tr):,}행 {fit_sec:.0f}s | test {len(te):,}행 / {len(v):,}방문 | "
          f"방문 acc {row['visit_acc']:.4f} | 행 acc {row['row_acc']:.4f} (빈 행 오답 {row['row_acc_empty_wrong']:.4f})")

## 3. Closed-world 실행

6회(A·B·C × 포함·제외). B·C는 학습 행이 7만~9만 개라 조건 하나에 10~16분 걸린다.

In [ ]:
for cond in ["A", "B", "C"]:
    for ver in ["include", "exclude"]:
        run("closed", cond, ver)

[closed A include] 학습 15,200행 226s | test 19,000행 / 3,800방문 | 방문 acc 0.0121 | 행 acc 0.0128 (빈 행 오답 0.0128)
[closed A exclude] 학습 15,200행 226s | test 17,342행 / 3,800방문 | 방문 acc 0.0139 | 행 acc 0.0135 (빈 행 오답 0.0123)
[closed B include] 학습 76,000행 627s | test 19,000행 / 3,800방문 | 방문 acc 0.1655 | 행 acc 0.0675 (빈 행 오답 0.0675)
[closed B exclude] 학습 69,696행 559s | test 17,342행 / 3,800방문 | 방문 acc 0.1668 | 행 acc 0.0693 (빈 행 오답 0.0632)
[closed C include] 학습 91,200행 976s | test 19,000행 / 3,800방문 | 방문 acc 0.1397 | 행 acc 0.0623 (빈 행 오답 0.0623)
[closed C exclude] 학습 84,896행 906s | test 17,342행 / 3,800방문 | 방문 acc 0.1397 | 행 acc 0.0633 (빈 행 오답 0.0578)


## 4. (선택) Open-world 이진

0번 설정에서 `RUN_BINARY = True`일 때만 실행된다. extra는 mon:unmon = 1:1이라 메인 이진 수치와 직접 비교하지 않는다.

In [ ]:
if RUN_BINARY:
    for cond in ["A", "B", "C"]:
        for ver in ["include", "exclude"]:
            run("binary", cond, ver)
else:
    print("RUN_BINARY = False → 건너뜀")

RUN_BINARY = False → 건너뜀


## 5. 결과 요약

In [ ]:
s = load_summary()
s = s[s.model == "lr"].copy()
cols = ["task", "condition", "version", "n_train_rows", "n_test_rows", "n_test_visits",
        "visit_acc", "visit_macro_f1", "row_acc", "row_acc_empty_wrong", "fit_sec"]
display(s[cols].sort_values(["task", "condition", "version"]).round(4).reset_index(drop=True))

for task in s.task.unique():
    print(f"\n[{task}] 방문 단위 정확도 (행: 조건, 열: 버전)")
    display(s[s.task == task].pivot(index="condition", columns="version", values="visit_acc").round(4))
print("\n참고: 메인 Closed-world LR test 정확도 0.6884 (일반 트래픽, 방문 하나 = 기록 하나)")

,task,condition,version,n_train_rows,n_test_rows,n_test_visits,visit_acc,visit_macro_f1,row_acc,row_acc_empty_wrong,fit_sec
0,closed,A,exclude,15200,17342,3800,0.0139,0.0034,0.0135,0.0123,225.8
1,closed,A,include,15200,19000,3800,0.0121,0.0022,0.0128,0.0128,225.8
2,closed,B,exclude,69696,17342,3800,0.1668,0.1373,0.0693,0.0632,559.3
3,closed,B,include,76000,19000,3800,0.1655,0.1353,0.0675,0.0675,626.5
4,closed,C,exclude,84896,17342,3800,0.1397,0.1177,0.0633,0.0578,906.2
5,closed,C,include,91200,19000,3800,0.1397,0.1152,0.0623,0.0623,976.4



[closed] 방문 단위 정확도 (행: 조건, 열: 버전)


version,exclude,include
condition,,
A,0.0139,0.0121
B,0.1668,0.1655
C,0.1397,0.1397



참고: 메인 Closed-world LR test 정확도 0.6884 (일반 트래픽, 방문 하나 = 기록 하나)


## 6. 저장 파일

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:45s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:6.1f} MB")

ec_summary.csv                                   0.0 MB
ec_test_proba_closed_A_exclude.csv              19.9 MB
ec_test_proba_closed_A_include.csv              21.9 MB
ec_test_proba_closed_B_exclude.csv              18.2 MB
ec_test_proba_closed_B_include.csv              19.9 MB
ec_test_proba_closed_C_exclude.csv              18.2 MB
ec_test_proba_closed_C_include.csv              20.0 MB
